# 5D KL random fields

One paired realization, four adaptive methods, fixed p=10, N=3010 to 8000 in batches of 10. Full execution regenerates a 4.8 GB candidate-basis cache. Only the small smoke mode changes these settings.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import os, sys, json
import numpy as np
HERE = Path.cwd()
if not (HERE.parent / 'PCA_Theta.py').exists():
    HERE = HERE / 'RandomField_5D'
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))
from PCA_Theta import METHODS, run_2d, save_result, load_result
from scipy import linalg
from PCA_Theta import prepare_matrix_cache, run_kl
from threadpoolctl import threadpool_limits

## 1. KL field definition

In [2]:
from __future__ import annotations

from itertools import product

import numpy as np
from numpy.polynomial.legendre import leggauss


EXAMPLE_NAME = "rf5d_ell04"
INPUT_DIMENSION = 5
N_OUTPUTS = 20
BOUNDS = [(0.0, 1.0)] * INPUT_DIMENSION


def make_target_corr() -> np.ndarray:
    """Four five-output groups with a stable low-rank correlation structure."""
    group_sizes = (5, 5, 5, 5)
    within = (0.95, 0.95, 0.85, 0.85)
    between = 0.05
    starts = np.cumsum((0,) + group_sizes)
    groups = [range(starts[g], starts[g + 1]) for g in range(4)]
    correlation = np.eye(sum(group_sizes))
    for g, indices in enumerate(groups):
        for i in indices:
            for j in indices:
                if i != j:
                    correlation[i, j] = within[g]
    for g in range(4):
        for h in range(g + 1, 4):
            for i in groups[g]:
                for j in groups[h]:
                    correlation[i, j] = correlation[j, i] = between
    eigenvalues = np.linalg.eigvalsh(correlation)
    if eigenvalues.min() <= 0.0:
        raise ValueError("The prescribed output-correlation matrix is not positive definite.")
    return correlation


def symmetric_square_root(matrix: np.ndarray) -> np.ndarray:
    values, vectors = np.linalg.eigh(matrix)
    if values.min() < -1.0e-10:
        raise ValueError("The target output-correlation matrix is not positive semidefinite.")
    return vectors @ np.diag(np.sqrt(np.maximum(values, 0.0))) @ vectors.T


def latin_hypercube(seed: int, n_samples: int, dimension: int) -> np.ndarray:
    rng = np.random.default_rng(int(seed))
    samples = np.empty((int(n_samples), int(dimension)), dtype=float)
    for axis in range(int(dimension)):
        permutation = rng.permutation(int(n_samples))
        samples[:, axis] = (permutation + rng.random(int(n_samples))) / int(n_samples)
    return samples


def one_dimensional_kl(length: float, quadrature_order: int, n_modes: int):
    nodes_raw, weights_raw = leggauss(int(quadrature_order))
    nodes = 0.5 * (nodes_raw + 1.0)
    weights = 0.5 * weights_raw
    difference = nodes[:, None] - nodes[None, :]
    kernel = np.exp(-(difference**2) / (2.0 * float(length) ** 2))
    sqrt_weights = np.sqrt(weights)
    operator = sqrt_weights[:, None] * kernel * sqrt_weights[None, :]
    eigenvalues, eigenvectors_weighted = np.linalg.eigh(operator)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = np.maximum(eigenvalues[order], 0.0)
    eigenvectors_weighted = eigenvectors_weighted[:, order]
    positive = eigenvalues > 1.0e-14
    eigenvalues = eigenvalues[positive][: int(n_modes)]
    eigenvectors_weighted = eigenvectors_weighted[:, positive][:, : int(n_modes)]
    eigenfunctions_at_nodes = eigenvectors_weighted / sqrt_weights[:, None]
    return {
        "nodes": nodes,
        "weights": weights,
        "eigenvalues": eigenvalues,
        "eigenfunctions_at_nodes": eigenfunctions_at_nodes,
        "length": float(length),
    }


def tensor_kl_terms(kl_1d, dimension: int, variance_threshold: float):
    values_1d = kl_1d["eigenvalues"]
    records = []
    for multi_index in product(range(len(values_1d)), repeat=int(dimension)):
        value = float(np.prod(values_1d[list(multi_index)]))
        records.append((value, multi_index))
    records.sort(key=lambda item: item[0], reverse=True)
    retained = []
    cumulative = 0.0
    for value, multi_index in records:
        retained.append((value, multi_index))
        cumulative += value
        if cumulative >= float(variance_threshold):
            break
    eigenvalues = np.asarray([item[0] for item in retained], dtype=float)
    multi_indices = np.asarray([item[1] for item in retained], dtype=np.int16)
    return eigenvalues, multi_indices, float(eigenvalues.sum())


def evaluate_one_dimensional_modes(coordinates, kl_1d):
    coordinates = np.asarray(coordinates, dtype=float).reshape(-1)
    nodes = kl_1d["nodes"]
    weights = kl_1d["weights"]
    eigenvalues = kl_1d["eigenvalues"]
    eigenfunctions = kl_1d["eigenfunctions_at_nodes"]
    length = kl_1d["length"]
    kernel_cross = np.exp(
        -((coordinates[:, None] - nodes[None, :]) ** 2) / (2.0 * length**2)
    )
    return kernel_cross @ (weights[:, None] * eigenfunctions) / eigenvalues[None, :]


def evaluate_tensor_modes(X, kl_1d, multi_indices):
    X = np.atleast_2d(np.asarray(X, dtype=float))
    one_dimensional = [
        evaluate_one_dimensional_modes(X[:, axis], kl_1d)
        for axis in range(X.shape[1])
    ]
    modes = np.ones((len(X), len(multi_indices)), dtype=float)
    for term, multi_index in enumerate(multi_indices):
        for axis in range(X.shape[1]):
            modes[:, term] *= one_dimensional[axis][:, multi_index[axis]]
    return modes


def build_model(args):
    dimension = int(args.dimension)
    n_outputs = int(args.n_outputs)
    if dimension != INPUT_DIMENSION or n_outputs != N_OUTPUTS:
        raise ValueError("This benchmark is fixed at five inputs and 20 outputs.")

    kl_1d = one_dimensional_kl(
        length=args.correlation_length,
        quadrature_order=args.kl_quadrature_order,
        n_modes=args.kl_modes_1d,
    )
    eigenvalues, multi_indices, captured_variance = tensor_kl_terms(
        kl_1d=kl_1d,
        dimension=dimension,
        variance_threshold=args.kl_variance_threshold,
    )

    target_correlation = make_target_corr()
    root_correlation = symmetric_square_root(target_correlation)
    rng = np.random.default_rng(int(args.field_seed))
    independent = rng.standard_normal((len(eigenvalues), n_outputs))
    correlated = independent @ root_correlation.T
    coefficient_matrix = np.sqrt(eigenvalues)[:, None] * correlated

    calibration = latin_hypercube(
        seed=int(args.field_seed) + 9183,
        n_samples=args.calibration_size,
        dimension=dimension,
    )
    calibration_modes = evaluate_tensor_modes(calibration, kl_1d, multi_indices)
    calibration_raw = calibration_modes @ coefficient_matrix
    field_mean = calibration_raw.mean(axis=0)
    field_std = np.maximum(
        calibration_raw.std(axis=0, ddof=1), np.finfo(float).tiny
    )

    def evaluate(X):
        X = np.atleast_2d(np.asarray(X, dtype=float))
        if X.shape[1] != dimension:
            raise ValueError(f"Expected {dimension} columns, received {X.shape[1]}.")
        modes = evaluate_tensor_modes(X, kl_1d, multi_indices)
        raw = modes @ coefficient_matrix
        return (raw - field_mean) / field_std

    metadata = {
        "dimension": dimension,
        "n_outputs": n_outputs,
        "correlation_length": float(args.correlation_length),
        "kl_variance_threshold": float(args.kl_variance_threshold),
        "kl_captured_variance": captured_variance,
        "kl_terms": int(len(eigenvalues)),
        "kl_eigenvalues": eigenvalues,
        "kl_multi_indices": multi_indices,
        "target_correlation": target_correlation,
        "target_correlation_eigenvalues": np.linalg.eigvalsh(target_correlation)[::-1],
        "field_seed": int(args.field_seed),
        "coefficient_matrix": coefficient_matrix,
        "field_mean": field_mean,
        "field_std": field_std,
    }
    return evaluate, metadata

## 2. Settings

In [3]:
RUN_MODE = 'reference'  # reference, smoke, or full
METHODS_TO_RUN = ['theta', 'pca_theta', 'eta_theta', 'eta_pca_theta']
CONFIG = dict(dimension=5, n_outputs=20, correlation_length=0.4,
              kl_variance_threshold=0.99, kl_quadrature_order=80, kl_modes_1d=8,
              p=10, n_init=3010, n_final=8000, batch_size=10,
              n_candidates=200000, n_test=5000, pca_threshold=0.01)
OUTPUT = Path(os.environ.get('PCA_THETA_OUTPUT', HERE/'Results'))

## 3. Shared inputs and the paper field realization

The stored inputs preserve the exact realization and LHS candidate plan used for the published comparison. Changing the kernel or field parameters requires regenerating these inputs.

In [4]:
with np.load(HERE/'Data/inputs.npz', allow_pickle=False) as saved:
    inputs = {k: saved[k] for k in ('initial_X','initial_Y','test_X','test_Y','candidates')}
    metadata = {k.removeprefix('meta_'): saved[k] for k in saved.files if k.startswith('meta_')}

# Align only the signs of the KL eigenfunctions across LAPACK implementations.
# The archived modal coefficients and response realization are unchanged.
one_dim = one_dimensional_kl(CONFIG['correlation_length'], CONFIG['kl_quadrature_order'], CONFIG['kl_modes_1d'])
def tensor_modes(x):
    return evaluate_tensor_modes(x, one_dim, metadata['kl_multi_indices'])
coefficients = metadata['coefficient_matrix']
mean, std = metadata['field_mean'], metadata['field_std']
with threadpool_limits(limits=1):
    observed = linalg.lstsq(tensor_modes(inputs['initial_X'][:1200]),
        inputs['initial_Y'][:1200]*std+mean, lapack_driver='gelsy')[0]
    signs = np.where(np.sum(observed*coefficients, axis=1)>=0, 1., -1.)
    aligned = signs[:,None]*coefficients
    np.testing.assert_allclose(aligned, observed, rtol=1e-12, atol=1e-12)
def evaluate(x):
    return (tensor_modes(x)@aligned-mean)/std
np.testing.assert_allclose(evaluate(inputs['initial_X']), inputs['initial_Y'], rtol=1e-12, atol=1e-12)
print('KL modes:', metadata['kl_terms'], '| candidates:', len(inputs['candidates']))

KL modes: 262 | candidates: 200000


## 4. Run or resume

In [5]:
if RUN_MODE not in ('reference','smoke','full'):
    raise ValueError('Choose reference, smoke, or full.')
if RUN_MODE == 'reference':
    print('Using the four complete paper trajectories in Data/. No simulation started.')
else:
    config = dict(CONFIG)
    working_inputs = dict(inputs)
    if RUN_MODE == 'smoke':
        config.update(p=2,n_init=64,n_final=74,batch_size=10,n_candidates=256,n_test=64)
        working_inputs = {k: v[:config['n_candidates'] if k=='candidates' else
                                   config['n_init'] if k.startswith('initial') else config['n_test']]
                          for k,v in inputs.items()}
    matrices = prepare_matrix_cache(OUTPUT/RUN_MODE/'Cache', working_inputs, config['p'])
    for method in METHODS_TO_RUN:
        def progress(stage):
            if stage['n']%100 == 0 or stage['n'] in (config['n_init'],config['n_final']):
                print(f"{stage['method']}: N={stage['n']}, NRMSE={stage['nrmse']:.5g}",flush=True)
        result=run_kl(evaluate,working_inputs,matrices,config,method,
                      OUTPUT/RUN_MODE/(method+'.npz'),metadata=metadata,progress=progress)
        print(method,'complete at',result['hist'][-1]['n'])

Using the four complete paper trajectories in Data/. No simulation started.


## 5. Reference results

In [6]:
for method in METHODS_TO_RUN:
    result=load_result(HERE/'Data'/(method+'.npz'))
    assert result['complete'] and result['hist'][-1]['n']==8000
    cpu=result['total_cpu']-result['cpu_timing']['test_error_evaluation']
    print(f"{method:18s} NRMSE={result['hist'][-1]['nrmse']:.5g}, construction CPU={cpu:.1f} s")

theta              NRMSE=0.0041113, construction CPU=4300.8 s


pca_theta          NRMSE=0.0039088, construction CPU=3873.1 s
eta_theta          NRMSE=0.0037737, construction CPU=4124.5 s


eta_pca_theta      NRMSE=0.0039507, construction CPU=3874.0 s
